In [8]:
#Load The Dataset

In [9]:
import numpy as np 
import pandas as pd

In [10]:
df=pd.read_csv("Dataset.csv",encoding="latin-1")

In [11]:
df.head(2)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820


In [18]:
#check for Null and Duplicated Values as well as Dataset errors
df.info()                                # data types and non-empty counts
print()

print("Null Values")
print(df.isnull().sum())                 # missing values per column
print()

print("Duplicated Values")
print("Duplicates:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

In [19]:
#Formatting Column Names 
df.columns = df.columns.str.lower().str.replace(" ", "_").str.replace("-", "_")
print(list(df.columns))

['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


In [26]:
#Format Date Types
df["order_date"] = pd.to_datetime(df["order_date"], format="%m/%d/%Y", errors="coerce")
df["ship_date"] = pd.to_datetime(df["ship_date"], format="%m/%d/%Y", errors="coerce")

In [31]:
#Fix Postal Codes
df["postal_code"] = df["postal_code"].astype(str).str.zfill(5)
df["postal_code"]

0       42420
1       42420
2       90036
3       33311
4       33311
        ...  
9989    33180
9990    92627
9991    92627
9992    92627
9993    92683
Name: postal_code, Length: 9994, dtype: object

In [33]:
#Cleaning Cleaning Text Values
# Remove extra spaces at the start and end
text_columns = ["order_id", "customer_id", "customer_name", "segment", "region",
                "state", "city", "category", "sub_category", "product_name"]
for col in text_columns:
    df[col] = df[col].str.strip()

# Make customer IDs consistent (upper case)
df["customer_id"] = df["customer_id"].str.upper()

# Fix two names with lower-case words
df["customer_name"] = df["customer_name"].replace({"Dorris liebe": "Dorris Liebe",
                                                   "Dean percer": "Dean Percer"})

In [34]:
df.columns

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'sub_category',
       'product_name', 'sales', 'quantity', 'discount', 'profit'],
      dtype='object')

In [39]:
#Handling Missing Values
print("Missing values:", df.isnull().sum().sum())
#If a columns has missing values example discount column-->
# df["discount"] = df["discount"].fillna(0)

Missing values: 0


In [42]:
#Handling Duplicated Values
print("Duplicates:", df.duplicated().sum())

# Only if duplicates exist: keep the first copy and remove the rest
df = df.drop_duplicates()

#Check for order_id columns as multiple customers can order same item with same order_id

Duplicates: 0


In [45]:
#Removing Invalid Rows
print("Sales of 0 or less:", (df["sales"] <= 0).sum())
print("Quantity of 0 or less:", (df["quantity"] <= 0).sum())
print("Discount outside 0 to 1:", ((df["discount"] < 0) | (df["discount"] > 1)).sum())

# If any count above is not 0, you could remove those rows like this:
df = df[df["sales"] > 0]
df = df[df["quantity"] > 0]
df = df[(df["discount"] >= 0) & (df["discount"] <= 1)]

Sales of 0 or less: 0
Quantity of 0 or less: 0
Discount outside 0 to 1: 0


In [47]:
#Checking Negative Profits 
#They are real loss-making sales caused by heavy discounts
print("Negative profit rows:", (df["profit"] < 0).sum())   # 1,871 rows, keep them

Negative profit rows: 1871


In [48]:
#Final Check for dataset
# Sort by date and reset the row numbers
df = df.sort_values(["order_date", "order_id", "row_id"]).reset_index(drop=True)

# Confirm data types are now correct
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   row_id         9994 non-null   int64         
 1   order_id       9994 non-null   object        
 2   order_date     9994 non-null   datetime64[ns]
 3   ship_date      9994 non-null   datetime64[ns]
 4   ship_mode      9994 non-null   object        
 5   customer_id    9994 non-null   object        
 6   customer_name  9994 non-null   object        
 7   segment        9994 non-null   object        
 8   country        9994 non-null   object        
 9   city           9994 non-null   object        
 10  state          9994 non-null   object        
 11  postal_code    9994 non-null   object        
 12  region         9994 non-null   object        
 13  product_id     9994 non-null   object        
 14  category       9994 non-null   object        
 15  sub_category   9994 n

In [ ]:
#Reference values for